# BR001 | Cleaning & Revalidation

## 01 — Setup & Source Baseline

**Objective:** Establish a reproducible source baseline before implementing approved cleaning decisions.

**Source:** Nova Analytics Warehouse (SQLite, read-only).

**Governance:** Business Clarification Response V3 — CL-001 to CL-007.

**Rule:** Preserve original data. Apply approved treatments in a separate analytical layer.

In [42]:
import sqlite3
import pandas as pd
from pathlib import Path

# Project configuration.
def find_project_root():
    current = Path.cwd().resolve()

    for directory in [current, *current.parents]:
        if (
            (directory / "04_data").is_dir()
            and (directory / "05_analysis").is_dir()
        ):
            return directory

    raise FileNotFoundError(
        "Nova Corporation project root not found."
    )


PROJECT_ROOT = find_project_root()

# Source and output paths.
WAREHOUSE_PATH = PROJECT_ROOT / "04_data" / "nova_analytics_warehouse.db"
CLEAN_DIR = PROJECT_ROOT / "04_data" / "clean"

# Validate source.
assert WAREHOUSE_PATH.is_file(), "Source database not found."

# Connect in read-only mode.
conn = sqlite3.connect(
    WAREHOUSE_PATH.resolve().as_uri() + "?mode=ro",
    uri=True
)

print("Source connection: SUCCESS")
print("Warehouse:", WAREHOUSE_PATH.name)
print("Read-only mode: ENABLED")

Source connection: SUCCESS
Warehouse: nova_analytics_warehouse.db
Read-only mode: ENABLED


In [43]:
# List all source tables.
tables = pd.read_sql_query(
    """
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    conn
)

# Inspect the sales table structure.
sales_schema = pd.read_sql_query(
    "PRAGMA table_info(fact_sales)",
    conn
)

display(tables)
display(sales_schema[["name", "type", "notnull", "pk"]])

,table_name
0,analyst_available_objects
1,column_catalog
2,dim_channel
3,dim_customer
4,dim_date
5,dim_product
6,dim_region
7,fact_sales
8,fact_sales_targets
9,warehouse_catalog


,name,type,notnull,pk
0,sales_line_id,TEXT,0,0
1,order_id,TEXT,0,0
2,order_date,TEXT,0,0
3,customer_id,TEXT,0,0
4,product_id,TEXT,0,0
5,channel_id,TEXT,0,0
6,region_id,TEXT,0,0
7,quantity,INTEGER,0,0
8,unit_price,REAL,0,0
9,unit_cost,REAL,0,0


In [44]:
# Establish the original sales baseline.
baseline = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS source_rows,
        COUNT(DISTINCT sales_line_id) AS distinct_sales_line_ids,
        COUNT(DISTINCT order_id) AS distinct_order_ids,
        ROUND(SUM(net_revenue), 2) AS recorded_net_revenue,
        ROUND(SUM(total_cost), 2) AS recorded_total_cost,
        ROUND(SUM(gross_profit), 2) AS recorded_gross_profit
    FROM fact_sales
    """,
    conn
)

display(baseline)

# Validate the expected source population.
assert baseline.loc[0, "source_rows"] == 23530, (
    "Unexpected source population. Stop and investigate."
)

print("Source population validation: PASS")

,source_rows,distinct_sales_line_ids,distinct_order_ids,recorded_net_revenue,recorded_total_cost,recorded_gross_profit
0,23530,23527,14803,14937536.82,10442161.65,4495375.17


Source population validation: PASS


In [45]:
# Identify repeated sales line identifiers.
duplicate_ids = pd.read_sql_query(
    """
    SELECT
        sales_line_id,
        COUNT(*) AS occurrences
    FROM fact_sales
    GROUP BY sales_line_id
    HAVING COUNT(*) > 1
    ORDER BY sales_line_id
    """,
    conn
)

display(duplicate_ids)

# Validate the documented DQ005 finding.
assert len(duplicate_ids) == 3
assert duplicate_ids["occurrences"].sum() == 6

print("DQ005 / CL-004 baseline: PASS")

,sales_line_id,occurrences
0,SL0000251,2
1,SL0008001,2
2,SL0017001,2


DQ005 / CL-004 baseline: PASS


## 02 — Cleaning Rules & Traceability

**Objective:** Translate approved business clarification decisions into
controlled, reproducible cleaning rules.

**Decision authority:** BR001 Business Clarification Response V3.

**Source baseline:** 23,530 sales lines.

**Implementation principle:** Preserve source values, apply analytical
treatments separately, and validate every transformation before acceptance.

In [46]:
# Approved cleaning decisions.
cleaning_rules = pd.DataFrame([
    {
        "cleaning_id": "CR-001",
        "clarification_id": "CL-004",
        "dq_reference": "DQ005",
        "treatment": "Preserve records and assign stable analytical row identity",
        "validation": "Unique row keys; no record loss",
    },
    {
        "cleaning_id": "CR-002",
        "clarification_id": "CL-006",
        "dq_reference": "DQ004 / DQ009",
        "treatment": "Retain missing references with Unknown attribution",
        "validation": "No row loss; dimensional totals reconcile",
    },
    {
        "cleaning_id": "CR-003",
        "clarification_id": "CL-003",
        "dq_reference": "DQ007",
        "treatment": "Preserve source payment method; classify unmapped values",
        "validation": "Approved domain; financial totals unchanged",
    },
    {
        "cleaning_id": "CR-004",
        "clarification_id": "CL-005",
        "dq_reference": "DQ007",
        "treatment": "Flag invalid numeric records and control KPI eligibility",
        "validation": "Numeric rules pass for eligible population",
    },
    {
        "cleaning_id": "CR-005",
        "clarification_id": "CL-002",
        "dq_reference": "DQ015",
        "treatment": "Apply approved transaction-status eligibility",
        "validation": "Status populations and financial totals reconcile",
    },
    {
        "cleaning_id": "CR-006",
        "clarification_id": "CL-007",
        "dq_reference": "DQ013",
        "treatment": "Retain valid outliers and flag negative margins",
        "validation": "Outlier baseline and eligible population reconcile",
    },
    {
        "cleaning_id": "CR-007",
        "clarification_id": "CL-001",
        "dq_reference": "Order integrity investigation",
        "treatment": "Preserve order IDs; restrict unverified order KPIs",
        "validation": "Order conflicts reproduced; no uncertified KPI published",
    },
])

cleaning_rules["implementation_status"] = "NOT STARTED"
cleaning_rules["validation_status"] = "PENDING"

display(cleaning_rules)

,cleaning_id,clarification_id,dq_reference,treatment,validation,implementation_status,validation_status
0,CR-001,CL-004,DQ005,Preserve records and assign stable analytical ...,Unique row keys; no record loss,NOT STARTED,PENDING
1,CR-002,CL-006,DQ004 / DQ009,Retain missing references with Unknown attribu...,No row loss; dimensional totals reconcile,NOT STARTED,PENDING
2,CR-003,CL-003,DQ007,Preserve source payment method; classify unmap...,Approved domain; financial totals unchanged,NOT STARTED,PENDING
3,CR-004,CL-005,DQ007,Flag invalid numeric records and control KPI e...,Numeric rules pass for eligible population,NOT STARTED,PENDING
4,CR-005,CL-002,DQ015,Apply approved transaction-status eligibility,Status populations and financial totals reconcile,NOT STARTED,PENDING
5,CR-006,CL-007,DQ013,Retain valid outliers and flag negative margins,Outlier baseline and eligible population recon...,NOT STARTED,PENDING
6,CR-007,CL-001,Order integrity investigation,Preserve order IDs; restrict unverified order ...,Order conflicts reproduced; no uncertified KPI...,NOT STARTED,PENDING


In [47]:
# Confirm the cleaning register is complete and consistent.
expected_clarifications = {
    "CL-001", "CL-002", "CL-003", "CL-004",
    "CL-005", "CL-006", "CL-007"
}

assert len(cleaning_rules) == 7
assert cleaning_rules["cleaning_id"].is_unique
assert cleaning_rules["clarification_id"].is_unique

assert set(cleaning_rules["clarification_id"]) == expected_clarifications

assert (
    cleaning_rules["implementation_status"] == "NOT STARTED"
).all()

assert (
    cleaning_rules["validation_status"] == "PENDING"
).all()

print("Cleaning Decision Register: PASS")
print("Approved decisions registered:", len(cleaning_rules))
print("Implementation status: NOT STARTED")

Cleaning Decision Register: PASS
Approved decisions registered: 7
Implementation status: NOT STARTED


## 03 — Record Identity | CL-004

**Cleaning ID:** CR-001  
**DQ Reference:** DQ005  
**Approved Decision:** Preserve all sales records and assign a stable analytical row identity.

**Objective:** Resolve technical row identification without deleting or modifying source transactions.

In [48]:
# CL-004: Inspect SQLite row identity.
row_identity_check = pd.read_sql_query(
    """
    SELECT
        rowid AS source_rowid,
        sales_line_id,
        order_id,
        order_date,
        customer_id,
        net_revenue
    FROM fact_sales
    ORDER BY rowid
    """,
    conn
)

display(row_identity_check.head())

print("Source rows:", len(row_identity_check))
print("Unique rowids:", row_identity_check["source_rowid"].nunique())
print("Null rowids:", row_identity_check["source_rowid"].isna().sum())

assert len(row_identity_check) == 23530
assert row_identity_check["source_rowid"].is_unique
assert row_identity_check["source_rowid"].notna().all()

print("CL-004 row identity assessment: PASS")

,source_rowid,sales_line_id,order_id,order_date,customer_id,net_revenue
0,1,SL0000001,O0000001,2025-02-11,C00874,789.86
1,2,SL0000002,O0000002,2025-04-27,C01154,678.86
2,3,SL0000003,O0000002,2025-03-24,C01565,40.35
3,4,SL0000004,O0000002,2025-04-24,C02110,205.30
4,5,SL0000005,O0000003,2025-02-07,C00466,377.98


Source rows: 23530
Unique rowids: 23530
Null rowids: 0
CL-004 row identity assessment: PASS


In [49]:
import hashlib

def calculate_sha256(file_path):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            sha256.update(chunk)

    return sha256.hexdigest()


source_snapshot_hash = calculate_sha256(WAREHOUSE_PATH)

print("Source database:", WAREHOUSE_PATH.name)
print("Source SHA-256:", source_snapshot_hash)

Source database: nova_analytics_warehouse.db
Source SHA-256: 180c6f846d151ea0684f3040cc07d4c67e9efdf5d3f64b8d61d8c73094896502


In [50]:
# Load source sales with their SQLite row identity.
df_clean = pd.read_sql_query(
    """
    SELECT
        rowid AS source_rowid,
        *
    FROM fact_sales
    ORDER BY rowid
    """,
    conn
)

# Preserve the original business identifier.
df_clean["sales_line_id_source"] = df_clean["sales_line_id"]

# Create a snapshot-scoped technical key.
df_clean["analytical_row_key"] = (
    source_snapshot_hash[:16]
    + "-"
    + df_clean["source_rowid"].astype(str).str.zfill(6)
)

# Identify all repeated source sales-line identifiers.
df_clean["line_id_collision_flag"] = (
    df_clean["sales_line_id_source"].duplicated(keep=False)
)

display(
    df_clean.loc[
        df_clean["line_id_collision_flag"],
        [
            "analytical_row_key",
            "source_rowid",
            "sales_line_id_source",
            "order_id",
            "net_revenue"
        ]
    ].sort_values("sales_line_id_source")
)

,analytical_row_key,source_rowid,sales_line_id_source,order_id,net_revenue
250,180c6f846d151ea0-000251,251,SL0000251,O0000169,180.21
251,180c6f846d151ea0-000252,252,SL0000251,O0000170,104.54
8000,180c6f846d151ea0-008001,8001,SL0008001,O0005090,148.82
8001,180c6f846d151ea0-008002,8002,SL0008001,O0005091,2539.68
17000,180c6f846d151ea0-017001,17001,SL0017001,O0010657,782.33
17001,180c6f846d151ea0-017002,17002,SL0017001,O0010658,1022.64


In [51]:
# CR-001: Record Identity Revalidation

source_rows = int(baseline.loc[0, "source_rows"])

assert len(df_clean) == source_rows
assert df_clean["analytical_row_key"].is_unique
assert df_clean["analytical_row_key"].notna().all()

assert df_clean["line_id_collision_flag"].sum() == 6

assert (
    df_clean["sales_line_id_source"]
    == df_clean["sales_line_id"]
).all()

# Reconcile financial values against the source baseline.
financial_columns = {
    "net_revenue": "recorded_net_revenue",
    "total_cost": "recorded_total_cost",
    "gross_profit": "recorded_gross_profit"
}

for column, baseline_column in financial_columns.items():
    original_total = float(baseline.loc[0, baseline_column])
    analytical_total = round(float(df_clean[column].sum()), 2)

    assert abs(original_total - analytical_total) <= 0.01, (
        f"Financial reconciliation failed: {column}"
    )

print("CR-001 / CL-004 REVALIDATION: PASS")
print("Rows preserved:", len(df_clean))
print("Unique analytical keys:", df_clean["analytical_row_key"].nunique())
print("Collision rows flagged:", df_clean["line_id_collision_flag"].sum())
print("Financial reconciliation: PASS")

CR-001 / CL-004 REVALIDATION: PASS
Rows preserved: 23530
Unique analytical keys: 23530
Collision rows flagged: 6
Financial reconciliation: PASS


## 04 — Dimension & Domain Treatment

### CR-002 | Missing Dimension References (CL-006)

**Approved Decision:** Preserve records with missing customer or channel
references and assign explicit Unknown reporting attribution.

**Objective:** Prevent unintended record loss during dimensional joins
without fabricating source identifiers.

In [52]:
# CR-002: Identify missing customer and channel references.
missing_references = df_clean.loc[
    df_clean["customer_id"].isna()
    | df_clean["channel_id"].isna(),
    [
        "analytical_row_key",
        "sales_line_id",
        "customer_id",
        "channel_id",
        "net_revenue"
    ]
]

display(missing_references)

print(
    "Missing customer references:",
    df_clean["customer_id"].isna().sum()
)

print(
    "Missing channel references:",
    df_clean["channel_id"].isna().sum()
)

,analytical_row_key,sales_line_id,customer_id,channel_id,net_revenue
104,180c6f846d151ea0-000105,SL0000105,NaN,CH04,215.95
8421,180c6f846d151ea0-008422,SL0008422,C02226,NaN,158.24


Missing customer references: 1
Missing channel references: 1


In [53]:
# Preserve source identifiers and flag missing references.
df_clean["missing_customer_flag"] = (
    df_clean["customer_id"].isna()
)

df_clean["missing_channel_flag"] = (
    df_clean["channel_id"].isna()
)

# Add reporting attribution without changing original keys.
df_clean["customer_attribution"] = (
    df_clean["customer_id"].fillna("Unknown Customer")
)

df_clean["channel_attribution"] = (
    df_clean["channel_id"].fillna("Unknown Channel")
)

In [54]:
# CR-002 / CL-006 — Missing Dimension Revalidation

assert len(df_clean) == 23530

assert df_clean["missing_customer_flag"].sum() == 1
assert df_clean["missing_channel_flag"].sum() == 1

assert (
    df_clean.loc[
        df_clean["missing_customer_flag"],
        "customer_attribution"
    ] == "Unknown Customer"
).all()

assert (
    df_clean.loc[
        df_clean["missing_channel_flag"],
        "channel_attribution"
    ] == "Unknown Channel"
).all()

assert df_clean["customer_attribution"].notna().all()
assert df_clean["channel_attribution"].notna().all()

# Verify original identifiers remain unchanged.
source_keys = pd.read_sql_query(
    """
    SELECT
        rowid AS source_rowid,
        customer_id,
        channel_id
    FROM fact_sales
    """,
    conn
)

comparison = df_clean[
    ["source_rowid", "customer_id", "channel_id"]
].merge(
    source_keys,
    on="source_rowid",
    suffixes=("_clean", "_source"),
    validate="one_to_one"
)

for field in ["customer_id", "channel_id"]:
    assert comparison[f"{field}_clean"].equals(
        comparison[f"{field}_source"]
    )

print("CR-002 / CL-006 REVALIDATION: PASS")
print("Rows preserved:", len(df_clean))
print("Missing customers:", df_clean["missing_customer_flag"].sum())
print("Missing channels:", df_clean["missing_channel_flag"].sum())
print("Original identifiers: PRESERVED")

CR-002 / CL-006 REVALIDATION: PASS
Rows preserved: 23530
Missing customers: 1
Missing channels: 1
Original identifiers: PRESERVED


In [55]:
# CR-003 / CL-003 — Original payment method investigation

payment_methods = pd.read_sql_query(
    """
    SELECT
        payment_method,
        COUNT(*) AS sales_lines
    FROM fact_sales
    GROUP BY payment_method
    ORDER BY sales_lines DESC
    """,
    conn
)

display(payment_methods)

display(
    df_clean.loc[
        df_clean["sales_line_id"] == "SL0020778",
        ["sales_line_id", "payment_method", "order_id"]
    ]
)

,payment_method,sales_lines
0,Bank Transfer,4775
1,Debit Card,4768
2,Digital Wallet,4758
3,Credit Card,4662
4,Cash,4566
5,Credit Young Card,1


,sales_line_id,payment_method,order_id
20777,SL0020778,Credit Young Card,O0013076


In [56]:
# Inspect payment-method values present in the source.
source_payment_domain = (
    df_clean["payment_method"]
    .value_counts(dropna=False)
    .rename_axis("payment_method")
    .reset_index(name="sales_lines")
)

display(source_payment_domain)

,payment_method,sales_lines
0,Bank Transfer,4775
1,Debit Card,4768
2,Digital Wallet,4758
3,Credit Card,4662
4,Cash,4566
5,Credit Young Card,1


In [57]:
# CR-003 / CL-003 — Approved payment method classification

valid_payment_methods = {
    "Bank Transfer",
    "Debit Card",
    "Digital Wallet",
    "Credit Card",
    "Cash"
}

# Preserve the original source value.
df_clean["payment_method_source"] = df_clean["payment_method"]

# Flag values outside the approved reporting domain.
df_clean["unmapped_payment_flag"] = (
    ~df_clean["payment_method_source"].isin(valid_payment_methods)
)

# Apply the approved analytical classification.
df_clean["payment_method_reporting"] = (
    df_clean["payment_method_source"]
    .where(
        ~df_clean["unmapped_payment_flag"],
        "Unmapped"
    )
)

# Review the reporting distribution.
payment_reporting_summary = (
    df_clean["payment_method_reporting"]
    .value_counts(dropna=False)
    .rename_axis("payment_method_reporting")
    .reset_index(name="sales_lines")
)

display(payment_reporting_summary)

,payment_method_reporting,sales_lines
0,Bank Transfer,4775
1,Debit Card,4768
2,Digital Wallet,4758
3,Credit Card,4662
4,Cash,4566
5,Unmapped,1


In [58]:
# CR-003 / CL-003 — Payment Method Revalidation

# Preserve the source population.
assert len(df_clean) == 23530

# Verify source values remain unchanged.
assert df_clean["payment_method_source"].equals(
    df_clean["payment_method"]
)

# Confirm the documented exception.
assert df_clean["unmapped_payment_flag"].sum() == 1

exception = df_clean.loc[
    df_clean["sales_line_id"] == "SL0020778"
]

assert len(exception) == 1
assert exception.iloc[0]["payment_method_source"] == "Credit Young Card"
assert exception.iloc[0]["payment_method_reporting"] == "Unmapped"

# Ensure all reporting values belong to the allowed domain.
assert df_clean["payment_method_reporting"].isin(
    valid_payment_methods | {"Unmapped"}
).all()

# Confirm that the transformation has not changed financial totals.
for column, baseline_column in financial_columns.items():
    source_total = float(baseline.loc[0, baseline_column])
    analytical_total = round(float(df_clean[column].sum()), 2)

    assert abs(source_total - analytical_total) <= 0.01, (
        f"Financial reconciliation failed: {column}"
    )

print("CR-003 / CL-003 REVALIDATION: PASS")
print("Rows preserved:", len(df_clean))
print("Unmapped payment methods:", df_clean["unmapped_payment_flag"].sum())
print("Original payment values: PRESERVED")
print("Financial reconciliation: PASS")

CR-003 / CL-003 REVALIDATION: PASS
Rows preserved: 23530
Unmapped payment methods: 1
Original payment values: PRESERVED
Financial reconciliation: PASS


## 05 — Numeric Quality & Eligibility

### CR-004 | Invalid Numeric Values (CL-005)

**DQ Reference:** DQ007

**Approved Decision:** Preserve original numeric values, flag invalid
transactions, and exclude unresolved numeric exceptions from certified KPIs.

**Objective:** Ensure financial reporting uses transactions that satisfy
the approved numeric business rules.

**Source Treatment:** No automatic correction or replacement.

In [59]:
# CR-004 / CL-005 — Original numeric exceptions

known_invalid_ids = [
    "SL0012001",
    "SL0015101",
    "SL0018301"
]

numeric_columns = [
    "sales_line_id",
    "order_status",
    "quantity",
    "unit_price",
    "unit_cost",
    "discount_pct",
    "net_revenue",
    "total_cost",
    "gross_profit"
]

known_invalid_records = df_clean.loc[
    df_clean["sales_line_id"].isin(known_invalid_ids),
    numeric_columns
].sort_values("sales_line_id")

display(known_invalid_records)

,sales_line_id,order_status,quantity,unit_price,unit_cost,discount_pct,net_revenue,total_cost,gross_profit
12000,SL0012001,Completed,-2,502.70,330.31,0.00,-1005.40,-660.62,-344.78
15100,SL0015101,Completed,1,626.96,-4.75,0.10,564.26,-4.75,569.01
18300,SL0018301,Completed,2,319.04,199.58,1.15,-95.71,399.16,-494.87


In [60]:
# CR-004 / CL-005 — Numeric quality assessment

numeric_rules = {
    "quantity": lambda s: s <= 0,
    "unit_price": lambda s: s <= 0,
    "unit_cost": lambda s: s < 0,
    "discount_pct": lambda s: (s < 0) | (s > 1)
}

numeric_fields = list(numeric_rules)

# Convert only for validation; preserve source columns.
numeric_check = df_clean[numeric_fields].apply(
    pd.to_numeric,
    errors="coerce"
)

# Flag missing, non-numeric, or out-of-range values.
for field, rule in numeric_rules.items():
    df_clean[f"invalid_{field}_flag"] = (
        numeric_check[field].isna()
        | rule(numeric_check[field])
    )

# Consolidate all numeric violations into one indicator.
numeric_flag_columns = [
    f"invalid_{field}_flag"
    for field in numeric_fields
]

df_clean["invalid_numeric_flag"] = (
    df_clean[numeric_flag_columns].any(axis=1)
)

# Numeric eligibility is one component of KPI eligibility.
df_clean["numeric_quality_eligible"] = (
    ~df_clean["invalid_numeric_flag"]
)

numeric_exceptions = df_clean.loc[
    df_clean["invalid_numeric_flag"],
    numeric_columns + numeric_flag_columns
]

display(numeric_exceptions)

print(
    "Invalid numeric records:",
    df_clean["invalid_numeric_flag"].sum()
)

,sales_line_id,order_status,quantity,unit_price,unit_cost,discount_pct,net_revenue,total_cost,gross_profit,invalid_quantity_flag,invalid_unit_price_flag,invalid_unit_cost_flag,invalid_discount_pct_flag
12000,SL0012001,Completed,-2,502.70,330.31,0.00,-1005.40,-660.62,-344.78,True,False,False,False
15100,SL0015101,Completed,1,626.96,-4.75,0.10,564.26,-4.75,569.01,False,False,True,False
18300,SL0018301,Completed,2,319.04,199.58,1.15,-95.71,399.16,-494.87,False,False,False,True


Invalid numeric records: 3


In [61]:
# CR-004 / CL-005 — Numeric Quality Revalidation

# 1. Preserve all source records.
assert len(df_clean) == 23530

# 2. Reproduce the documented exceptions.
assert df_clean["invalid_numeric_flag"].sum() == 3

detected_invalid_ids = set(
    df_clean.loc[
        df_clean["invalid_numeric_flag"],
        "sales_line_id"
    ]
)

assert detected_invalid_ids == set(known_invalid_ids)

# 3. Confirm the known invalid values.
expected_violations = {
    "SL0012001": ("quantity", -2),
    "SL0015101": ("unit_cost", -4.75),
    "SL0018301": ("discount_pct", 1.15)
}

for line_id, (field, expected_value) in expected_violations.items():
    record = df_clean.loc[
        df_clean["sales_line_id"] == line_id
    ]

    assert len(record) == 1
    assert abs(float(record.iloc[0][field]) - expected_value) < 1e-9
    assert bool(record.iloc[0][f"invalid_{field}_flag"])

# 4. Ensure eligible records satisfy every numeric rule.
assert df_clean.loc[
    df_clean["numeric_quality_eligible"],
    numeric_flag_columns
].sum().sum() == 0

# 5. Reconcile unfiltered financial totals.
for column, baseline_column in financial_columns.items():
    source_total = float(baseline.loc[0, baseline_column])
    analytical_total = round(float(df_clean[column].sum()), 2)

    assert abs(source_total - analytical_total) <= 0.01, (
        f"Financial reconciliation failed: {column}"
    )

# 6. Confirm numeric population reconciliation.
assert (
    df_clean["numeric_quality_eligible"].sum()
    + df_clean["invalid_numeric_flag"].sum()
    == len(df_clean)
)

print("CR-004 / CL-005 REVALIDATION: PASS")
print("Rows preserved:", len(df_clean))
print("Invalid numeric records:", df_clean["invalid_numeric_flag"].sum())
print("Numerically eligible records:", df_clean["numeric_quality_eligible"].sum())
print("Original financial totals: PRESERVED")
print("Numeric eligibility reconciliation: PASS")

CR-004 / CL-005 REVALIDATION: PASS
Rows preserved: 23530
Invalid numeric records: 3
Numerically eligible records: 23527
Original financial totals: PRESERVED
Numeric eligibility reconciliation: PASS


## 06 — Transaction Status & KPI Eligibility

### CR-005 | CL-002 | DQ015

**Objective:** Establish certified and provisional sales populations.

**Certified:** Completed transactions with valid numeric values.

**Provisional:** Completed and Partially Returned transactions
with valid numeric values.

**Source Treatment:** Preserve all records and original financial values.

**Limitation:** Return adjustments remain unverified.

In [62]:
# CR-005 / CL-002 — Transaction Status Baseline

status_baseline = pd.read_sql_query(
    """
    SELECT
        order_status,
        COUNT(*) AS sales_lines,
        ROUND(SUM(net_revenue), 2) AS net_revenue,
        ROUND(SUM(total_cost), 2) AS total_cost,
        ROUND(SUM(gross_profit), 2) AS gross_profit
    FROM fact_sales
    GROUP BY order_status
    ORDER BY order_status
    """,
    conn
)

display(status_baseline)

print("Total source records:", status_baseline["sales_lines"].sum())

assert status_baseline["sales_lines"].sum() == len(df_clean)

assert set(status_baseline["order_status"]) == {
    "Completed",
    "Partially Returned",
    "Pending",
    "Cancelled"
}

print("CR-005 STATUS BASELINE: PASS")

,order_status,sales_lines,net_revenue,total_cost,gross_profit
0,Cancelled,767,488760.04,339203.15,149556.89
1,Completed,21345,13563744.00,9483372.33,4080371.67
2,Partially Returned,1075,673506.50,470229.86,203276.64
3,Pending,343,211526.28,149356.31,62169.97


Total source records: 23530
CR-005 STATUS BASELINE: PASS


In [63]:
# CR-005 / CL-002 — Transaction Eligibility

# Preserve original transaction status
df_clean["order_status_source"] = df_clean["order_status"]

# Certified Sales: Completed + valid numeric values
df_clean["certified_sales_eligible"] = (
    df_clean["order_status_source"].eq("Completed")
    & df_clean["numeric_quality_eligible"]
)

# Provisional Sales: Completed + Partially Returned
# with valid numeric values
df_clean["provisional_return_inclusive_eligible"] = (
    df_clean["order_status_source"].isin(
        ["Completed", "Partially Returned"]
    )
    & df_clean["numeric_quality_eligible"]
)

# Eligibility distribution
eligibility_summary = (
    df_clean.groupby("order_status_source", dropna=False)
    .agg(
        source_lines=("sales_line_id", "size"),
        invalid_numeric_lines=("invalid_numeric_flag", "sum"),
        certified_lines=("certified_sales_eligible", "sum"),
        provisional_lines=(
            "provisional_return_inclusive_eligible", "sum"
        )
    )
    .reset_index()
)

display(eligibility_summary)

print(
    "Certified eligible lines:",
    df_clean["certified_sales_eligible"].sum()
)

print(
    "Provisional eligible lines:",
    df_clean["provisional_return_inclusive_eligible"].sum()
)

,order_status_source,source_lines,invalid_numeric_lines,certified_lines,provisional_lines
0,Cancelled,767,0,0,0
1,Completed,21345,3,21342,21342
2,Partially Returned,1075,0,0,1075
3,Pending,343,0,0,0


Certified eligible lines: 21342
Provisional eligible lines: 22417


In [64]:
# CR-005 / CL-002 — Financial Reconciliation

financial_metrics = [
    "net_revenue",
    "total_cost",
    "gross_profit"
]

population_masks = {
    "Source — All Records": pd.Series(True, index=df_clean.index),
    "Certified — Eligible": df_clean["certified_sales_eligible"],
    "Certified — Excluded": ~df_clean["certified_sales_eligible"],
    "Provisional — Eligible": df_clean["provisional_return_inclusive_eligible"],
    "Provisional — Excluded": ~df_clean["provisional_return_inclusive_eligible"]
}

reconciliation_rows = []

for population, mask in population_masks.items():
    selected = df_clean.loc[mask]

    reconciliation_rows.append({
        "population": population,
        "sales_lines": len(selected),
        **{
            metric: round(selected[metric].sum(), 2)
            for metric in financial_metrics
        }
    })

financial_reconciliation = pd.DataFrame(reconciliation_rows)

display(financial_reconciliation)

,population,sales_lines,net_revenue,total_cost,gross_profit
0,Source — All Records,23530,14937536.82,10442161.65,4495375.17
1,Certified — Eligible,21342,13564280.85,9483638.54,4080642.31
2,Certified — Excluded,2188,1373255.97,958523.11,414732.86
3,Provisional — Eligible,22417,14237787.35,9953868.40,4283918.95
4,Provisional — Excluded,1113,699749.47,488293.25,211456.22


In [65]:
# CR-005 / CL-002 — Final Revalidation

# 1. Source population preserved
assert len(df_clean) == 23530

# 2. Original transaction statuses preserved
assert df_clean["order_status_source"].equals(
    df_clean["order_status"]
)

# 3. Certified population: Completed + valid numerics
assert df_clean["certified_sales_eligible"].equals(
    df_clean["order_status_source"].eq("Completed")
    & ~df_clean["invalid_numeric_flag"]
)

# 4. Provisional population: Completed / Partially Returned
assert df_clean["provisional_return_inclusive_eligible"].equals(
    df_clean["order_status_source"].isin(
        ["Completed", "Partially Returned"]
    )
    & ~df_clean["invalid_numeric_flag"]
)

# 5. Certified population must be a subset of provisional
assert not (
    df_clean["certified_sales_eligible"]
    & ~df_clean["provisional_return_inclusive_eligible"]
).any()

# 6. Expected eligible populations
assert df_clean["certified_sales_eligible"].sum() == 21342
assert df_clean["provisional_return_inclusive_eligible"].sum() == 22417

# 7. Partial-return population
assert df_clean["order_status_source"].eq(
    "Partially Returned"
).sum() == 1075

# 8. Financial reconciliation
for metric in ["net_revenue", "total_cost", "gross_profit"]:
    source_total = df_clean[metric].sum()

    certified_total = df_clean.loc[
        df_clean["certified_sales_eligible"], metric
    ].sum()

    certified_excluded = df_clean.loc[
        ~df_clean["certified_sales_eligible"], metric
    ].sum()

    provisional_total = df_clean.loc[
        df_clean["provisional_return_inclusive_eligible"], metric
    ].sum()

    provisional_excluded = df_clean.loc[
        ~df_clean["provisional_return_inclusive_eligible"], metric
    ].sum()

    assert abs(
        source_total - certified_total - certified_excluded
    ) < 0.01

    assert abs(
        source_total - provisional_total - provisional_excluded
    ) < 0.01

print("CR-005 / CL-002 FINAL REVALIDATION: PASS")
print("Source records:", len(df_clean))
print("Certified eligible:", df_clean["certified_sales_eligible"].sum())
print(
    "Provisional eligible:",
    df_clean["provisional_return_inclusive_eligible"].sum()
)
print("Original transaction statuses: PRESERVED")
print("Financial reconciliation: PASS")

CR-005 / CL-002 FINAL REVALIDATION: PASS
Source records: 23530
Certified eligible: 21342
Provisional eligible: 22417
Original transaction statuses: PRESERVED
Financial reconciliation: PASS


## 07 — Outlier & Negative Margin Treatment

### CR-006 | CL-007 | DQ013

**Objective:** Preserve and identify high-cost outliers and
negative gross profit transactions.

**Approved Decision:** Retain valid transactions without
automatic cost or profit adjustments.

**Methodology:** Reproduce the documented DQ013 statistical
thresholds before implementing outlier flags.

**Governance:** Statistical anomalies remain subject to
business review.

In [66]:
# CR-006 / CL-007 — Negative Gross Profit Baseline

negative_margin_baseline = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS negative_margin_lines,
        ROUND(SUM(net_revenue), 2) AS net_revenue,
        ROUND(SUM(total_cost), 2) AS total_cost,
        ROUND(SUM(gross_profit), 2) AS gross_profit
    FROM fact_sales
    WHERE gross_profit < 0
    """,
    conn
)

display(negative_margin_baseline)

print(
    "Negative gross profit lines:",
    negative_margin_baseline.loc[0, "negative_margin_lines"]
)

,negative_margin_lines,net_revenue,total_cost,gross_profit
0,1735,659612.94,721854.43,-62241.49


Negative gross profit lines: 1735


In [67]:
# CR-006 / CL-007 — Negative Margin by Transaction Status

negative_margin_by_status = pd.read_sql_query(
    """
    SELECT
        order_status,
        COUNT(*) AS negative_margin_lines,
        ROUND(SUM(net_revenue), 2) AS net_revenue,
        ROUND(SUM(total_cost), 2) AS total_cost,
        ROUND(SUM(gross_profit), 2) AS gross_profit
    FROM fact_sales
    WHERE gross_profit < 0
    GROUP BY order_status
    ORDER BY order_status
    """,
    conn
)

display(negative_margin_by_status)

assert negative_margin_by_status["negative_margin_lines"].sum() == 1735

print("Negative margin distribution: RECONCILED")

,order_status,negative_margin_lines,net_revenue,total_cost,gross_profit
0,Cancelled,64,22022.99,23826.32,-1803.33
1,Completed,1555,589143.87,645775.93,-56632.06
2,Partially Returned,79,34358.88,36975.87,-2616.99
3,Pending,37,14087.20,15276.31,-1189.11


Negative margin distribution: RECONCILED


In [68]:
# CR-006 / CL-007 — Negative Margin by Analytical Population

negative_margin_summary = pd.DataFrame([
    {
        "population": "Source — All Records",
        "negative_margin_lines": (
            df_clean["gross_profit"] < 0
        ).sum(),
        "negative_gross_profit": round(
            df_clean.loc[
                df_clean["gross_profit"] < 0,
                "gross_profit"
            ].sum(), 2
        )
    },
    {
        "population": "Certified — Eligible",
        "negative_margin_lines": (
            df_clean["certified_sales_eligible"]
            & (df_clean["gross_profit"] < 0)
        ).sum(),
        "negative_gross_profit": round(
            df_clean.loc[
                df_clean["certified_sales_eligible"]
                & (df_clean["gross_profit"] < 0),
                "gross_profit"
            ].sum(), 2
        )
    },
    {
        "population": "Provisional — Eligible",
        "negative_margin_lines": (
            df_clean["provisional_return_inclusive_eligible"]
            & (df_clean["gross_profit"] < 0)
        ).sum(),
        "negative_gross_profit": round(
            df_clean.loc[
                df_clean["provisional_return_inclusive_eligible"]
                & (df_clean["gross_profit"] < 0),
                "gross_profit"
            ].sum(), 2
        )
    }
])

display(negative_margin_summary)

,population,negative_margin_lines,negative_gross_profit
0,Source — All Records,1735,-62241.49
1,Certified — Eligible,1553,-55792.41
2,Provisional — Eligible,1632,-58409.40


In [69]:
# CR-006 / CL-007 — Outlier & Negative Margin Flags

# Reproduce DQ013 IQR using the original source population
source_total_cost = pd.read_sql_query(
    "SELECT total_cost FROM fact_sales",
    conn
)["total_cost"]

q1_cost = source_total_cost.quantile(0.25)
q3_cost = source_total_cost.quantile(0.75)

iqr_cost = q3_cost - q1_cost
upper_bound_cost = q3_cost + 1.5 * iqr_cost

# Preserve all original values; add analytical flags only
df_clean["high_total_cost_outlier_flag"] = (
    df_clean["total_cost"] > upper_bound_cost
)

df_clean["negative_gross_profit_flag"] = (
    df_clean["gross_profit"] < 0
)

df_clean["high_cost_negative_profit_flag"] = (
    df_clean["high_total_cost_outlier_flag"]
    & df_clean["negative_gross_profit_flag"]
)

# Summarize findings
outlier_summary = pd.DataFrame({
    "Control": [
        "High Total Cost Outliers",
        "Negative Gross Profit — All Records",
        "High Cost + Negative Gross Profit"
    ],
    "Records": [
        int(df_clean["high_total_cost_outlier_flag"].sum()),
        int(df_clean["negative_gross_profit_flag"].sum()),
        int(df_clean["high_cost_negative_profit_flag"].sum())
    ]
})

display(outlier_summary)

print("DQ013 Upper Bound:", round(upper_bound_cost, 5))

,Control,Records
0,High Total Cost Outliers,2610
1,Negative Gross Profit — All Records,1735
2,High Cost + Negative Gross Profit,190


DQ013 Upper Bound: 974.34125


In [70]:
# CR-006 / CL-007 — Final Revalidation

# 1. Source population preserved
assert len(df_clean) == 23530

# 2. Original DQ013 IQR threshold reproduced
assert abs(upper_bound_cost - 974.34125) < 0.00001

# 3. Original outlier population reproduced
assert df_clean["high_total_cost_outlier_flag"].sum() == 2610

# 4. Negative gross profit population
assert df_clean["negative_gross_profit_flag"].sum() == 1735

# 5. Original DQ013 intersection reproduced
assert df_clean["high_cost_negative_profit_flag"].sum() == 190

# 6. Verify intersection logic
assert df_clean["high_cost_negative_profit_flag"].equals(
    df_clean["high_total_cost_outlier_flag"]
    & df_clean["negative_gross_profit_flag"]
)

# 7. Verify Completed records within the intersection
completed_high_cost_negative_profit = df_clean.loc[
    df_clean["high_cost_negative_profit_flag"]
    & df_clean["order_status_source"].eq("Completed")
]

assert len(completed_high_cost_negative_profit) == 167

# 8. Preserve original financial totals
expected_financial_totals = {
    "net_revenue": 14937536.82,
    "total_cost": 10442161.65,
    "gross_profit": 4495375.17
}

for metric, expected in expected_financial_totals.items():
    actual = round(df_clean[metric].sum(), 2)
    assert abs(actual - expected) < 0.01

# 9. Preserve CR-005 eligibility populations
assert df_clean["certified_sales_eligible"].sum() == 21342
assert df_clean["provisional_return_inclusive_eligible"].sum() == 22417

print("CR-006 / CL-007 FINAL REVALIDATION: PASS")
print("Source records:", len(df_clean))
print("High total cost outliers:", df_clean["high_total_cost_outlier_flag"].sum())
print("Negative gross profit records:", df_clean["negative_gross_profit_flag"].sum())
print("High cost + negative profit:", df_clean["high_cost_negative_profit_flag"].sum())
print("Completed within intersection:", len(completed_high_cost_negative_profit))
print("Original financial totals: PRESERVED")
print("CR-005 eligibility populations: PRESERVED")

CR-006 / CL-007 FINAL REVALIDATION: PASS
Source records: 23530
High total cost outliers: 2610
Negative gross profit records: 1735
High cost + negative profit: 190
Completed within intersection: 167
Original financial totals: PRESERVED
CR-005 eligibility populations: PRESERVED


## 08 — Order Identity & Integrity

### CR-007 | CL-001

**Objective:** Preserve original order identifiers, identify
conflicting order-level attributes, and prevent unsupported
order-based KPI certification.

**Approved Decision:** Do not reconstruct or fabricate
business order identifiers.

**Technical Row Identity:** analytical_row_key (CR-001).

**Business Order Identity:** Unresolved pending authoritative mapping.

**Restricted KPIs:** Order Count and Average Order Value (AOV).

In [71]:
# CR-007 / CL-001 — Original Order Identity Assessment

order_identity_baseline = pd.read_sql_query(
    """
    WITH order_profiles AS (
        SELECT
            order_id,
            COUNT(*) AS sales_lines,
            COUNT(DISTINCT customer_id) AS distinct_customers,
            COUNT(DISTINCT order_date) AS distinct_dates
        FROM fact_sales
        GROUP BY order_id
    )
    SELECT
        COUNT(*) AS distinct_order_ids,

        SUM(
            CASE WHEN distinct_customers > 1
            THEN 1 ELSE 0 END
        ) AS multi_customer_order_ids,

        SUM(
            CASE WHEN distinct_dates > 1
            THEN 1 ELSE 0 END
        ) AS multi_date_order_ids

    FROM order_profiles
    """,
    conn
)

display(order_identity_baseline)

,distinct_order_ids,multi_customer_order_ids,multi_date_order_ids
0,14803,5518,5498


In [72]:
# CR-007 / CL-001 — Order Identity Conflict Flags

# Preserve original business order identifiers
df_clean["order_id_source"] = df_clean["order_id"]

# Calculate order-level profiles
order_profiles = (
    df_clean.groupby("order_id_source", dropna=False)
    .agg(
        distinct_customers=("customer_id", "nunique"),
        distinct_dates=("order_date", "nunique")
    )
)

# Identify conflicting order IDs
multi_customer_ids = order_profiles.index[
    order_profiles["distinct_customers"] > 1
]

multi_date_ids = order_profiles.index[
    order_profiles["distinct_dates"] > 1
]

# Apply conflict flags at sales-line level
df_clean["order_customer_conflict_flag"] = (
    df_clean["order_id_source"].isin(multi_customer_ids)
)

df_clean["order_date_conflict_flag"] = (
    df_clean["order_id_source"].isin(multi_date_ids)
)

df_clean["order_identity_conflict_flag"] = (
    df_clean["order_customer_conflict_flag"]
    | df_clean["order_date_conflict_flag"]
)

# Summarize findings
order_conflict_summary = pd.DataFrame({
    "Control": [
        "Distinct Order IDs",
        "Multi-Customer Order IDs",
        "Multi-Date Order IDs",
        "Sales Lines with Identity Conflicts",
        "Missing Order IDs"
    ],
    "Records": [
        df_clean["order_id_source"].nunique(),
        len(multi_customer_ids),
        len(multi_date_ids),
        int(df_clean["order_identity_conflict_flag"].sum()),
        int(df_clean["order_id_source"].isna().sum())
    ]
})

display(order_conflict_summary)

,Control,Records
0,Distinct Order IDs,14803
1,Multi-Customer Order IDs,5518
2,Multi-Date Order IDs,5498
3,Sales Lines with Identity Conflicts,14245
4,Missing Order IDs,0


In [73]:
# CR-007 / CL-001 — Composite Order Identity Assessment

composite_profiles = (
    df_clean.groupby(
        ["order_id_source", "channel_id", "region_id"],
        dropna=False
    )
    .agg(
        sales_lines=("analytical_row_key", "size"),
        distinct_customers=("customer_id", "nunique"),
        distinct_dates=("order_date", "nunique")
    )
    .reset_index()
)

# A composite group is conflicting when customer or date
# is inconsistent within the same candidate identity.
composite_profiles["composite_conflict_flag"] = (
    (composite_profiles["distinct_customers"] > 1)
    | (composite_profiles["distinct_dates"] > 1)
)

conflicting_composite_groups = composite_profiles.loc[
    composite_profiles["composite_conflict_flag"]
]

composite_summary = pd.DataFrame({
    "Control": [
        "Total Composite Groups",
        "Conflicting Composite Groups",
        "Sales Lines in Conflicting Groups"
    ],
    "Records": [
        len(composite_profiles),
        len(conflicting_composite_groups),
        int(conflicting_composite_groups["sales_lines"].sum())
    ]
})

display(composite_summary)

,Control,Records
0,Total Composite Groups,23067
1,Conflicting Composite Groups,446
2,Sales Lines in Conflicting Groups,909


In [74]:
# CR-007 / CL-001 — KPI Certification Governance

kpi_certification_register = pd.DataFrame([
    {
        "kpi": "Net Revenue",
        "certification_status": "ELIGIBLE FOR FINAL VALIDATION",
        "approved_population": "Certified Sales",
        "restriction": "Subject to integrated revalidation"
    },
    {
        "kpi": "Gross Profit",
        "certification_status": "ELIGIBLE FOR FINAL VALIDATION",
        "approved_population": "Certified Sales",
        "restriction": "Subject to integrated revalidation"
    },
    {
        "kpi": "Order Count",
        "certification_status": "BLOCKED",
        "approved_population": "None",
        "restriction": "Unresolved business order identity (CL-001)"
    },
    {
        "kpi": "Average Order Value",
        "certification_status": "BLOCKED",
        "approved_population": "None",
        "restriction": "Requires an approved Order Count definition (CL-001)"
    }
])

display(kpi_certification_register)

,kpi,certification_status,approved_population,restriction
0,Net Revenue,ELIGIBLE FOR FINAL VALIDATION,Certified Sales,Subject to integrated revalidation
1,Gross Profit,ELIGIBLE FOR FINAL VALIDATION,Certified Sales,Subject to integrated revalidation
2,Order Count,BLOCKED,None,Unresolved business order identity (CL-001)
3,Average Order Value,BLOCKED,None,Requires an approved Order Count definition (C...


In [80]:
# CR-007 / CL-001 — Final Revalidation

# Order KPI Certification Controls
df_clean["order_key_verified"] = False
df_clean["order_kpi_eligible"] = False
assert df_clean["order_key_verified"].eq(False).all()
assert df_clean["order_kpi_eligible"].eq(False).all()

# 1. Source population preserved
assert len(df_clean) == 23530

# 2. Original business identifiers preserved
assert df_clean["order_id_source"].equals(df_clean["order_id"])

# 3. Technical row identity remains valid
assert df_clean["analytical_row_key"].is_unique
assert df_clean["analytical_row_key"].notna().all()

# 4. Order identity findings reproduced
assert df_clean["order_id_source"].nunique() == 14803
assert len(multi_customer_ids) == 5518
assert len(multi_date_ids) == 5498

assert df_clean["order_identity_conflict_flag"].sum() == 14245

# 5. Composite candidate remains unresolved
assert len(conflicting_composite_groups) == 446
assert conflicting_composite_groups["sales_lines"].sum() == 909

# 6. Order-dependent KPIs remain blocked
blocked_kpis = set(
    kpi_certification_register.loc[
        kpi_certification_register["certification_status"] == "BLOCKED",
        "kpi"
    ]
)

assert {"Order Count", "Average Order Value"}.issubset(blocked_kpis)

# 7. Financial eligibility remains unchanged
assert df_clean["certified_sales_eligible"].sum() == 21342
assert df_clean["provisional_return_inclusive_eligible"].sum() == 22417

# 8. Original financial totals preserved
expected_totals = {
    "net_revenue": 14937536.82,
    "total_cost": 10442161.65,
    "gross_profit": 4495375.17
}

for metric, expected in expected_totals.items():
    assert abs(round(df_clean[metric].sum(), 2) - expected) < 0.01

print("CR-007 / CL-001 FINAL REVALIDATION: PASS")
print("Source records:", len(df_clean))
print("Distinct Order IDs:", df_clean["order_id_source"].nunique())
print("Conflicting sales lines:", df_clean["order_identity_conflict_flag"].sum())
print("Conflicting composite groups:", len(conflicting_composite_groups))
print("Order Count / AOV: BLOCKED")
print("Original identifiers: PRESERVED")
print("Financial populations: PRESERVED")
print("Dataset shape:", df_clean.shape)
print("Order key verified:", df_clean["order_key_verified"].sum())
print("Order KPI eligible:", df_clean["order_kpi_eligible"].sum())

CR-007 / CL-001 FINAL REVALIDATION: PASS
Source records: 23530
Distinct Order IDs: 14803
Conflicting sales lines: 14245
Conflicting composite groups: 446
Order Count / AOV: BLOCKED
Original identifiers: PRESERVED
Financial populations: PRESERVED
Dataset shape: (23530, 47)
Order key verified: 0
Order KPI eligible: 0


## 09 — Cleaning Dataset Handoff

**Objective:** Validate and export the final analytical
dataset produced by CR-001 through CR-007.

**Output:** br001_clean_analytical_dataset.csv

**Destination:** 04_data/clean/

**Certification Status:** PENDING INTEGRATED REVALIDATION

In [82]:
# ============================================================
# BR001 — CLEANING DATASET HANDOFF
# Controlled Export | CR-001 through CR-007
# ============================================================

import pandas as pd
from pathlib import Path
import hashlib

# ------------------------------------------------------------
# 1. Project Configuration
# ------------------------------------------------------------

PROJECT_ROOT = find_project_root()

CLEAN_DIR = PROJECT_ROOT / "04_data" / "clean"

CLEAN_DATASET_PATH = (
    CLEAN_DIR / "br001_clean_analytical_dataset.csv"
)

# ------------------------------------------------------------
# 2. Dataset Availability
# ------------------------------------------------------------

assert "df_clean" in globals(), (
    "df_clean is not available. "
    "Execute the Cleaning notebook before exporting."
)

assert isinstance(df_clean, pd.DataFrame), (
    "df_clean must be a pandas DataFrame."
)

# ------------------------------------------------------------
# 3. Population and Technical Identity
# ------------------------------------------------------------

assert len(df_clean) == 23530

assert df_clean["source_rowid"].is_unique
assert df_clean["source_rowid"].notna().all()

assert df_clean["analytical_row_key"].is_unique
assert df_clean["analytical_row_key"].notna().all()

# ------------------------------------------------------------
# 4. Required Cleaning Outputs
# ------------------------------------------------------------

required_columns = [

    # CR-001 — Record Identity
    "source_rowid",
    "analytical_row_key",
    "sales_line_id_source",
    "line_id_collision_flag",

    # CR-002 — Missing References
    "missing_customer_flag",
    "missing_channel_flag",
    "customer_attribution",
    "channel_attribution",

    # CR-003 — Payment Method
    "payment_method_source",
    "payment_method_reporting",
    "unmapped_payment_flag",

    # CR-004 — Numeric Quality
    "invalid_numeric_flag",
    "numeric_quality_eligible",

    # CR-005 — Transaction Eligibility
    "order_status_source",
    "certified_sales_eligible",
    "provisional_return_inclusive_eligible",

    # CR-006 — Outlier Diagnostics
    "high_total_cost_outlier_flag",
    "negative_gross_profit_flag",
    "high_cost_negative_profit_flag",

    # CR-007 — Order Integrity
    "order_id_source",
    "order_customer_conflict_flag",
    "order_date_conflict_flag",
    "order_identity_conflict_flag"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df_clean.columns
]

assert not missing_columns, (
    f"Missing Cleaning outputs: {missing_columns}"
)

# ------------------------------------------------------------
# 5. Cleaning Results Reconciliation
# ------------------------------------------------------------

expected_flags = {
    "line_id_collision_flag": 6,
    "missing_customer_flag": 1,
    "missing_channel_flag": 1,
    "unmapped_payment_flag": 1,
    "invalid_numeric_flag": 3,
    "high_total_cost_outlier_flag": 2610,
    "negative_gross_profit_flag": 1735,
    "high_cost_negative_profit_flag": 190,
    "order_identity_conflict_flag": 14245
}

for column, expected in expected_flags.items():

    actual = int(df_clean[column].sum())

    assert actual == expected, (
        f"{column}: expected {expected}, found {actual}"
    )

# ------------------------------------------------------------
# 6. KPI Eligibility Reconciliation
# ------------------------------------------------------------

certified_lines = int(
    df_clean["certified_sales_eligible"].sum()
)

provisional_lines = int(
    df_clean["provisional_return_inclusive_eligible"].sum()
)

assert certified_lines == 21342
assert provisional_lines == 22417

# ------------------------------------------------------------
# 7. Financial Reconciliation
# ------------------------------------------------------------

expected_totals = {
    "net_revenue": 14937536.82,
    "total_cost": 10442161.65,
    "gross_profit": 4495375.17
}

for column, expected in expected_totals.items():

    actual = round(df_clean[column].sum(), 2)

    assert abs(actual - expected) < 0.01, (
        f"Financial reconciliation failed: {column}. "
        f"Expected {expected}, found {actual}"
    )

# ------------------------------------------------------------
# 8. Controlled Export
# ------------------------------------------------------------

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# Preserve the previous controlled dataset version

from pathlib import Path
from shutil import copy2

assert df_clean.shape == (23530, 47)
assert df_clean["order_key_verified"].eq(False).all()
assert df_clean["order_kpi_eligible"].eq(False).all()

backup_path = Path(CLEAN_DATASET_PATH).with_name(
    "br001_clean_analytical_dataset_pre_CL001_update.csv"
)

if not backup_path.exists():
    copy2(CLEAN_DATASET_PATH, backup_path)

print("Previous dataset preserved:", backup_path.name)


df_clean.to_csv(
    CLEAN_DATASET_PATH,
    index=False,
    encoding="utf-8",
    float_format="%.15g"
)

# ------------------------------------------------------------
# 9. Export File Integrity — SHA-256
# ------------------------------------------------------------

sha256 = hashlib.sha256()

with open(CLEAN_DATASET_PATH, "rb") as file:

    for chunk in iter(
        lambda: file.read(1024 * 1024),
        b""
    ):
        sha256.update(chunk)

export_sha256 = sha256.hexdigest()

# ------------------------------------------------------------
# 10. Handoff Summary
# ------------------------------------------------------------

print("=" * 60)
print("BR001 CLEANING DATASET HANDOFF")
print("=" * 60)

print("Export Status: SUCCESS")
print("File:", CLEAN_DATASET_PATH.name)

print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))

print("Cleaning Rules: CR-001 through CR-007")
print("Source Records: PRESERVED")
print("Technical Identity: VALIDATED")
print("Financial Totals: RECONCILED")

print("Certified Sales Lines:", certified_lines)
print("Provisional Sales Lines:", provisional_lines)

print("SHA-256:", export_sha256)

print("-" * 60)
print("Dataset Certification: PENDING INTEGRATED REVALIDATION")
print("=" * 60)

Previous dataset preserved: br001_clean_analytical_dataset_pre_CL001_update.csv
BR001 CLEANING DATASET HANDOFF
Export Status: SUCCESS
File: br001_clean_analytical_dataset.csv
Rows: 23530
Columns: 47
Cleaning Rules: CR-001 through CR-007
Source Records: PRESERVED
Technical Identity: VALIDATED
Financial Totals: RECONCILED
Certified Sales Lines: 21342
Provisional Sales Lines: 22417
SHA-256: e6f4aa911d83db5e4e315dd7c2f5e66933e506807d105b1e7522543dbb617de2
------------------------------------------------------------
Dataset Certification: PENDING INTEGRATED REVALIDATION


### Data Cleaning — Final Closure & Revalidation Handoff

**Project:** Nova Corporation — BR001 Executive Performance Review  
**Phase:** Data Cleaning  
**Scope:** CR-001 through CR-007  
**Implementation Status:** COMPLETED  
**Individual Validation Status:** PASS (7/7)  
**Dataset Export Status:** SUCCESS  
**Handoff Status:** READY FOR INTEGRATED REVALIDATION

#### Execution Summary

All seven cleaning treatments were implemented and individually
validated against the approved business clarification decisions.

The analytical dataset preserves all 23,530 source records,
original business identifiers, and recorded financial values.

The controlled export contains 45 columns and preserves
the documented cleaning flags and analytical eligibility rules.

#### Analytical Populations

- Source Sales Lines: 23,530
- Certified Sales Eligible Lines: 21,342
- Provisional Return-Inclusive Eligible Lines: 22,417

#### Outstanding Limitations

- Original source defects remain open unless formally resolved.
- Missing dimension references remain flagged.
- Financial outliers and negative-profit records remain available for review.
- Partially returned transactions require further business reconciliation.
- Order Count and Average Order Value remain blocked under CL-001.

#### Revalidation Handoff

**Dataset:** `br001_clean_analytical_dataset.csv`  
**Location:** `04_data/clean/`  
**Integrity Reference:** SHA-256 generated during controlled export  
**Next Phase:** Integrated Revalidation — IR-001 through IR-005

**Final Dataset Certification:** PENDING

Cleaning completion confirms implementation and individual validation
of approved analytical treatments. It does not constitute final dataset
certification or resolution of outstanding source defects.